In [3]:
# Necessary libraries
import os
import time # To measure loading time
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch
import matplotlib.pyplot as plt # For optional visualization
import numpy as np

In [4]:
root_dir = '/export/usuarios_ml4ds/danibacaicoa/ForwardBackard_losses_old/Datasets/raw_datasets/Clothing1M/' 

# Check if the directory exists
if not os.path.isdir(root_dir):
    print(f"Error: The specified root directory does not exist: {root_dir}")
    print("Please update the 'root_dir' variable in this cell.")
    # Stop execution if the path is wrong (in a real notebook, this would prevent subsequent cells from running correctly)
    assert False, "Root directory not found. Please fix the path."
else:
    print(f"Root directory set to: {root_dir}")

# Optional: Define class mapping if your labels are strings in kv files
# class_to_idx = {
#     'T-Shirt': 0, 'Shirt': 1, 'Knitwear': 2, 'Chiffon': 3, 'Sweater': 4,
#     'Anorak': 5, 'Hoodie': 6, 'Blazer': 7, 'Jacket': 8, 'Coat': 9,
#     'Dress': 10, 'Skirt': 11, 'Pants': 12, 'Shorts': 13
# }

Root directory set to: /export/usuarios_ml4ds/danibacaicoa/ForwardBackard_losses_old/Datasets/raw_datasets/Clothing1M/


In [5]:
def load_annotation_data(annotation_file_path, key_list_file_path=None, root_dir_path=root_dir):
    """
    Loads image paths and labels from Clothing1M annotation files.

    Args:
        annotation_file_path (str): Full path to the *_label_kv.txt file.
        key_list_file_path (str, optional): Full path to the *_key_list.txt file.
                                            Required for clean train/val/test sets.
        root_dir_path (str): The root directory of the Clothing1M dataset.

    Returns:
        list: A list of tuples, where each tuple is (full_image_path, label_int).
              Returns an empty list if files are not found or errors occur.
    """
    data = []
    image_keys = set()

    # Check if annotation file exists
    if not os.path.exists(annotation_file_path):
        print(f"Warning: Annotation file not found: {annotation_file_path}")
        return []

    # Load keys if a key list file is provided
    if key_list_file_path:
        if not os.path.exists(key_list_file_path):
            print(f"Warning: Key list file not found: {key_list_file_path}")
            return [] # Cannot filter without the key list
        with open(key_list_file_path, 'r') as f:
            image_keys = {line.strip() for line in f}
        print(f"  Loaded {len(image_keys)} keys from {os.path.basename(key_list_file_path)}")

    print(f"  Processing annotation file: {os.path.basename(annotation_file_path)}")
    processed_lines = 0
    skipped_label_conversion = 0
    skipped_key_mismatch = 0
    skipped_file_not_found = 0

    with open(annotation_file_path, 'r') as f:
        for line in f:
            processed_lines += 1
            parts = line.strip().split()
            if len(parts) != 2:
                # print(f"    Warning: Skipping malformed line: {line.strip()}")
                continue # Skip malformed lines

            image_key = parts[0] # e.g., "images/10000/12345.jpg" or just "12345.jpg" depending on kv file format
            label_str = parts[1]

            # Filter by key list if provided
            if key_list_file_path and image_key not in image_keys:
                skipped_key_mismatch += 1
                continue

            # Convert label to integer
            try:
                label = int(label_str)
                # Optional: Add validation for label range (e.g., 0-13 for 14 classes)
                # if not (0 <= label < 14):
                #    print(f"    Warning: Label {label} out of expected range (0-13) for key {image_key}. Skipping.")
                #    continue
            except ValueError:
                # Handle cases where label might be a class name string (if not using class_to_idx)
                # print(f"    Warning: Could not convert label '{label_str}' to int for key '{image_key}'. Skipping.")
                skipped_label_conversion += 1
                continue # Skip if label cannot be converted

            # Construct the full path to the image file
            # The exact structure might depend on how files were extracted.
            # Common structures:
            # 1. Images directly in root_dir: root_dir/image_key
            # 2. Images in subdirs (noisy_train, clean_train, etc.): root_dir/subdir/image_key
            # 3. image_key in kv file *includes* subdir: root_dir/image_key (where image_key is like 'noisy_train/abc.jpg')

            full_path = os.path.join(root_dir_path, image_key)

            if not os.path.exists(full_path):
                 # Try searching in common subdirectories if the direct path fails
                found = False
                common_subdirs = ['noisy_train', 'clean_train', 'clean_val', 'clean_test', 'images'] # Add 'images' if needed
                # Check if image_key already contains a path separator
                if os.path.dirname(image_key): # If image_key is like 'noisy_train/img.jpg'
                     potential_path = os.path.join(root_dir_path, image_key)
                     if os.path.exists(potential_path):
                         full_path = potential_path
                         found = True
                else: # If image_key is just 'img.jpg'
                    for subdir in common_subdirs:
                        potential_path = os.path.join(root_dir_path, subdir, image_key)
                        if os.path.exists(potential_path):
                            full_path = potential_path
                            found = True
                            break # Found it in a subdir

                if not found:
                    # print(f"    Warning: Image file not found for key '{image_key}' at expected paths. Skipping.")
                    skipped_file_not_found += 1
                    continue # Skip if image file doesn't exist

            data.append((full_path, label))

    print(f"  Finished processing. Total lines: {processed_lines}.")
    if key_list_file_path: print(f"    Skipped {skipped_key_mismatch} lines due to key mismatch.")
    if skipped_label_conversion > 0: print(f"    Skipped {skipped_label_conversion} lines due to label conversion errors.")
    if skipped_file_not_found > 0: print(f"    Skipped {skipped_file_not_found} lines because image file was not found.")
    print(f"  Successfully loaded {len(data)} samples.")
    return data

In [6]:
# --- Load all data components ---
start_time = time.time()

print("Loading noisy training data...")
noisy_train_file = os.path.join(root_dir, 'noisy_label_kv.txt')
noisy_train_data = load_annotation_data(noisy_train_file)

print("\nLoading clean training data...")
clean_label_file = os.path.join(root_dir, 'clean_label_kv.txt') # Shared by clean train/val/test
clean_train_key_file = os.path.join(root_dir, 'clean_train_key_list.txt')
clean_train_data = load_annotation_data(clean_label_file, clean_train_key_file)

# print("\nLoading clean validation data...") # Optional, uncomment if you need validation set
# clean_val_key_file = os.path.join(root_dir, 'clean_val_key_list.txt')
# clean_val_data = load_annotation_data(clean_label_file, clean_val_key_file)

print("\nLoading clean test data...")
clean_test_key_file = os.path.join(root_dir, 'clean_test_key_list.txt')
test_data_list = load_annotation_data(clean_label_file, clean_test_key_file) # Directly use as test_data_list

# --- Combine clean and noisy data for the final training list ---
train_data_list = []

# Add clean data with noisy_flag = 0
for path, label in clean_train_data:
    train_data_list.append((path, label, 0))

# Add noisy data with noisy_flag = 1
for path, label in noisy_train_data:
    train_data_list.append((path, label, 1))

end_time = time.time()

print("-" * 40)
print(f"Total noisy training samples: {len(noisy_train_data)}")
print(f"Total clean training samples: {len(clean_train_data)}")
# print(f"Total clean validation samples: {len(clean_val_data)}") # Uncomment if using validation set
print(f"Total clean test samples:     {len(test_data_list)}")
print(f"Combined training samples:  {len(train_data_list)}")
print(f"Data loading took: {end_time - start_time:.2f} seconds")
print("-" * 40)

# Basic sanity checks
assert len(train_data_list) > 0, "Training data list is empty! Check paths and file loading."
assert len(test_data_list) > 0, "Test data list is empty! Check paths and file loading."

Loading noisy training data...
  Processing annotation file: noisy_label_kv.txt
  Finished processing. Total lines: 1037497.
    Skipped 1037497 lines because image file was not found.
  Successfully loaded 0 samples.

Loading clean training data...
  Loaded 47570 keys from clean_train_key_list.txt
  Processing annotation file: clean_label_kv.txt
  Finished processing. Total lines: 72409.
    Skipped 24839 lines due to key mismatch.
    Skipped 47570 lines because image file was not found.
  Successfully loaded 0 samples.

Loading clean test data...
  Loaded 10526 keys from clean_test_key_list.txt
  Processing annotation file: clean_label_kv.txt
  Finished processing. Total lines: 72409.
    Skipped 61883 lines due to key mismatch.
    Skipped 10526 lines because image file was not found.
  Successfully loaded 0 samples.
----------------------------------------
Total noisy training samples: 0
Total clean training samples: 0
Total clean test samples:     0
Combined training samples:  0


AssertionError: Training data list is empty! Check paths and file loading.